# Ordered Logistic Regression Results for Adoption Predictors in Rangeland Management (FAIR^2) Exploration with `mlcroissant`
This notebook demonstrates the loading, exploration, and basic processing of the FAIR^2 dataset using the `mlcroissant` library. We strictly reference all dataset elements by their `@id` according to the Croissant schema specification.

### Dataset Source
Croissant schema URL: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset package using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the Croissant dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is an object, not a dict

print(f"Dataset name: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All references are via entity `@id` attributes per Croissant schema.

In [ ]:
# List all record sets with their @id, name, and available fields
print("Available record sets:")
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in the metadata. Please refer to the documentation or inspect the distributions.")
else:
    for rs in record_sets:
        print(f"- @id: {rs.id} | name: {getattr(rs, 'name', '[Unnamed]')}")
        if hasattr(rs, 'fields'):
            for f in rs.fields:
                print(f"    - Field @id: {f.id} | name: {getattr(f, 'name', '[Unnamed]')} | dataType: {getattr(f, 'data_type', '[Unknown]')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the shown `@id`s from the previous section.

> **Note:** In Croissant, primary data often appears as record sets. Here, we load all record sets present, referencing each by its `@id`.

In [ ]:
# Extract data from each record set using its @id
record_set_ids = [rs.id for rs in dataset.record_sets]
dfs_by_id = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dfs_by_id[rs_id] = pd.DataFrame(records)
    print(f"Loaded DataFrame for record set @id: {rs_id}")
    print(f"Columns: {dfs_by_id[rs_id].columns.tolist()}\nShape: {dfs_by_id[rs_id].shape}\n")

# Display the columns and preview head for the first available record set
if record_set_ids:
    primary_rs_id = record_set_ids[0]
    print(f"First record set @id: {primary_rs_id}")
    display(dfs_by_id[primary_rs_id].head())
else:
    print("No record sets available for extraction.")

## 4. Exploratory Data Analysis (EDA)
Perform standard filtering, normalization, and grouping using fields referenced **by their @id**.

In [ ]:
# For demonstration, pick first available record set and a numeric field (by @id)
EDA_rs_id = None
EDA_numeric_field_id = None
EDA_group_field_id = None

# Attempt to auto-select numeric and groupable fields
for rs in dataset.record_sets:
    numeric_fields = [f for f in getattr(rs, 'fields', []) if getattr(f, 'data_type', '').lower() in ["number", "integer", "float"]]
    group_fields = [f for f in getattr(rs, 'fields', []) if getattr(f, 'data_type', '').lower() in ["string", "text", "categorical"]]
    if numeric_fields:
        EDA_rs_id = rs.id
        EDA_numeric_field_id = numeric_fields[0].id
        if group_fields:
            EDA_group_field_id = group_fields[0].id
        break

if not EDA_rs_id or EDA_rs_id not in dfs_by_id:
    print("No suitable record set with numeric fields found for EDA.")
else:
    df = dfs_by_id[EDA_rs_id]
    if EDA_numeric_field_id not in df.columns:
        print(f"Numeric field @id '{EDA_numeric_field_id}' not found in DataFrame columns.")
    else:
        # Filtering (example: filter values > threshold)
        threshold = 10
        mask = pd.to_numeric(df[EDA_numeric_field_id], errors='coerce') > threshold
        filtered_df = df[mask].copy()
        print(f"Filtered records for {EDA_numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{EDA_numeric_field_id}_normalized"] = (pd.to_numeric(filtered_df[EDA_numeric_field_id], errors='coerce') - pd.to_numeric(filtered_df[EDA_numeric_field_id], errors='coerce').mean()) / pd.to_numeric(filtered_df[EDA_numeric_field_id], errors='coerce').std()
        print(f"Normalized field '{EDA_numeric_field_id}' for filtered records:")
        display(filtered_df[[EDA_numeric_field_id, f"{EDA_numeric_field_id}_normalized"]].head())

        # Grouping
        if EDA_group_field_id and EDA_group_field_id in df.columns:
            grouped_df = filtered_df.groupby(EDA_group_field_id)[EDA_numeric_field_id].mean().to_frame()
            print(f"Grouped mean of '{EDA_numeric_field_id}' by '{EDA_group_field_id}':")
            display(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships for the selected numeric field and grouping.

> All field and record set references are by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if EDA_rs_id and EDA_numeric_field_id and EDA_rs_id in dfs_by_id:
    df = dfs_by_id[EDA_rs_id]
    if EDA_numeric_field_id in df.columns:
        numeric_series = pd.to_numeric(df[EDA_numeric_field_id], errors='coerce')
        plt.figure(figsize=(8,4))
        sns.histplot(numeric_series.dropna(), kde=True, bins=30)
        plt.title(f"Distribution of Field @id: {EDA_numeric_field_id}")
        plt.xlabel(f"@id: {EDA_numeric_field_id}")
        plt.ylabel("Frequency")
        plt.show()

    if EDA_group_field_id and EDA_group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[EDA_group_field_id], y=numeric_series)
        plt.title(f"{EDA_numeric_field_id} by {EDA_group_field_id} (by @id)")
        plt.xlabel(f"@id: {EDA_group_field_id}")
        plt.ylabel(f"@id: {EDA_numeric_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load, overview, extract, and analyze a Croissant dataset using `mlcroissant`, referencing all entities by `@id` as required by the schema. The workflow outlined data filtering, normalization, grouping, and visualization to help explore ordered logistic regression results for rangeland management practice adoption predictors in Northern Kenya.

For further analysis, consider deeper statistical modeling, more advanced visualizations, or domain-specific investigation guided by the Croissant schema's semantic structure and field documentation.